# brain_spi — ABIDE quickstart

A compact tour of the two things `brain_spi` does, on the **ABIDE** fMRI ICA
dataset (controls vs. autism, 53 ICN components × 140 TRs):

* **Part 1 — compute SPIs and look at them**
* **Part 2 — the significant-differences pipeline**

The data downloads on first run. We use a **fast SPI subset** (the slow distance
SPIs — `dcorr`, `mgc`, `hsic` — are left out; add them back for a real analysis).
Part 1 uses a small balanced subsample for speed; Part 2 runs on the full ABIDE
cohort so there is enough power for edges to survive Bonferroni correction.

The loader also supports COBRE (schizophrenia) — set `DATASET = 'cobre'` below to
run the whole notebook on it instead.

In [ ]:
import sys, os, logging
sys.path.insert(0, '..')          # import brain_spi from the repo
sys.path.insert(0, '.')           # import the local datasets.py helper

import numpy as np
import matplotlib.pyplot as plt
logging.basicConfig(level=logging.INFO, format='%(message)s')  # per-SPI progress

import brain_spi
from brain_spi import BrainSPI
import brain_spi.domains as domains
from brain_spi.viz import plot_heatmap
import datasets   # local helper: download() + load()

DATASET = 'abide'   # or 'cobre'

# Domain guides for the 53 ICN components (shared by both datasets)
domain_spec = domains.from_csv('../data/ICN_coordinates.csv', name_col='Domain')
FAST_SPIS = ['cov_GraphicalLassoCV', 'spearmanr', 'kendalltau', 'xcorr_mean_sig-True', 'pec']
print('brain_spi ready  |  dataset:', DATASET, '|  SPIs:', FAST_SPIS)

In [ ]:
def balanced_subsample(data, labels, per_group=40, seed=0):
    """Take an equal number of subjects from each label (for a fast demo)."""
    rng = np.random.default_rng(seed)
    idx = np.concatenate([
        rng.choice(np.where(labels == c)[0], size=per_group, replace=False)
        for c in np.unique(labels)
    ])
    return data[idx], labels[idx]

datasets.download(DATASET)                  # cached after first run
data, labels, group_names = datasets.load(DATASET)
print(f'{DATASET}: data {data.shape}, labels {np.bincount(labels)}, groups {group_names}')

## Part 1 — Compute SPIs and look at them

`BrainSPI().fit()` computes one connectivity matrix per SPI per subject. Here we
just inspect the **connectivity itself** — the per-group mean matrices — without
any statistics yet. A small balanced subsample keeps it quick.

In [ ]:
d, y = balanced_subsample(data, labels, per_group=40)

pipe = BrainSPI(spis=FAST_SPIS, group_names=group_names)
result = pipe.fit(d, y)

result   # rich repr — shows SPIs, shapes, and points at .help()

In [ ]:
# Mean connectivity for one SPI, split by group
spi = 'kendalltau'
result[spi].plot_mean(by_group=True, group_names=group_names, domain_spec=domain_spec)
plt.show()

In [ ]:
# Overall mean connectivity across all SPIs, side by side
spis = result.spis
fig, axes = plt.subplots(1, len(spis), figsize=(3.2 * len(spis), 3.4))
for ax, name in zip(axes, spis):
    plot_heatmap(result[name].mean_matrix(), ax=ax, guides=domain_spec, title=name)
fig.suptitle(f'{DATASET.upper()} — mean connectivity per SPI', y=1.04)
fig.tight_layout()
plt.show()

## Part 2 — Significant-differences pipeline (full cohort)

For each SPI the pipeline runs a Welch t-test (Bonferroni-corrected → `p_thresh`)
and a Random-Forest importance mask (`rf_mask`), then intersects them (`and_mask`).
Averaging the AND masks across SPIs gives the cross-SPI **aggregate** — the
fraction of SPIs that flag each edge as discriminating between groups.

This runs on **all subjects** (the first fit takes a few minutes; results are
cached, so re-runs are fast).

In [ ]:
pipe_full = BrainSPI(spis=FAST_SPIS, group_names=group_names)
result_full = pipe_full.fit(data, labels)   # full cohort

# Per-SPI triptych: significant-p | RF importance | their AND
result_full['kendalltau'].plot_triptych(domain_spec=domain_spec)
plt.show()

In [ ]:
# Cross-SPI aggregate — computed lazily on first access
agg = result_full.aggregate
print('edges flagged by >=1 SPI:', int((agg.mean_and > 0).sum() // 2))
agg.plot(domain_spec=domain_spec)
plt.show()

In [ ]:
# Robustness: resample 2/3 of subjects 20x; how often is each edge flagged?
boot = result_full.bootstrap(n=20, frac=0.66)
survival = boot.survival_rate()   # fraction of resamples flagging each edge (0..1)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = plot_heatmap(survival, ax=ax, cmap='plasma', vmin=0, vmax=1,
                  guides=domain_spec, title=f'{DATASET.upper()} — bootstrap survival rate')
fig.colorbar(im, ax=ax, fraction=0.045)
plt.show()

# Save the full result for later (portable .npz — loads without brain_spi too)
result_full.to_npz(f'{DATASET}_result.npz')
print(f'saved -> {DATASET}_result.npz  |  reload with brain_spi.load_npz(...)')